## FlashAttention: Алгоритмическая оптимизация I/O в механизме Attention

**Аннотация:** *Механизм Self-Attention имеет квадратичную вычислительную и пространственную сложность $O(N^2)$, что ограничивает размер контекста при инференсе и обучении. В данном эссе рассматривается алгоритм FlashAttention, который снижает количество обращений к медленной памяти (I/O complexity) за счет блочного вычисления функции Softmax (tiling) и слияния ядер (kernel fusion).*

---

### 1. Проблема: квадратичная сложность Attention

<img src="multi-head.png" width="750">

Вычислительная сложность и потребление памяти стандартного механизма Attention растут пропорционально квадрату длины последовательности ($N^2$). При $N \gg d$ (где $d$ — размерность головы) накладные расходы на материализацию промежуточных матриц превышают объем доступной памяти ускорителей.

Методы аппроксимации (Sparse Attention, Low-Rank Approximations) снижают требования к памяти, но приводят к потере точности. Профилирование показывает, что основным bottleneck при вычислении exact Attention является не недостаток вычислительной мощности (FLOPs), а избыточный трафик между уровнями памяти.

---

### 2. Arithmetic Intensity и Memory-bound операции

Анализ аппаратной статистики показывает, что пиковая вычислительная производительность тензорных ядер растет существенно быстрее пропускной способности HBM. Отношение FLOPs к пропускной способности памяти (arithmetic intensity) на современных архитектурах требует выполнения $\sim 100-200$ операций на каждый загруженный байт для достижения полной утилизации ALU.

Операции в глубоком обучении делятся на:
* **Math-bound:** перемножение матриц большого размера. Высокая вычислительная интенсивность.
* **Memory-bound:** поэлементные операции (Softmax, Masking, Dropout). Вычислительная интенсивность низкая, время выполнения лимитировано пропускной способностью HBM.

*По статистике, производительность Tensor Cores в NVIDIA GPU растет примерно в 2 раза каждые 2 года, тогда как пропускная способность HBM памяти увеличивается лишь в 1.2 раза за тот же период (Gholami et al., 2021).*

Так вот оказывается, что в стандартном Attention значительная часть времени уходит на memory-bound операции.

---

### 3. Иерархия памяти ускорителей

Архитектура памяти вычислительных узлов имеет строгую иерархию, обусловленную физическими ограничениями (площадь кристалла, тепловыделение, задержка сигнала). Усредненные характеристики для современных ускорителей:

1. **Registers (Регистры):** Расположены в вычислительных блоках. Минимальная задержка (< 1 нс). Объем: $\sim 256$ КБ на мультипроцессор.
2. **SRAM (Shared Memory):** Разделяемая память на кристалле (on-chip). Пропускная способность: $\sim 10-20$ ТБ/с. Объем: $\sim 100-200$ КБ на мультипроцессор.
3. **HBM (High Bandwidth Memory):** Глобальная память устройства (off-chip). Пропускная способность: $\sim 1.5-3$ ТБ/с. Объем: десятки гигабайт.
4. **DRAM (Host Memory):** Системная память. Обмен данными идет через шину PCIe с пропускной способностью $\sim 10$ ГБ/с.

**Почему нельзя сделать GPU из SRAM или регистров?**
Кэш SRAM занимает большую физическую площадь на кристалле. Если сделать кристалл слишком большим, сигнал не успеет пройти от одного края до другого за один такт. Масштабирование объема SRAM на кристалле экономически и физически нецелесообразно, поэтому алгоритмы должны минимизировать обращения к HBM, максимально переиспользуя данные в SRAM и регистрах. Тем самым строится иерархия памяти.

<img src="main.png" width="1000">

---

### 4. I/O профиль стандартного Attention

Математически Self-Attention вычисляется как:
$$ \mathbf{O} = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d}}\right) \mathbf{V} $$
где $\mathbf{Q}, \mathbf{K}, \mathbf{V} \in \mathbb{R}^{N \times d}$, а $\mathbf{O} \in \mathbb{R}^{N \times d}$.

В стандартных фреймворках каждая операция реализуется отдельным вызовом ядра. Например, в PyTorch каждая базовая операция вызывает отдельный GPU Kernel — скомпилированную функцию, которая берет данные из медленной HBM, загружает в быструю SRAM, вычисляет результат и пишет его обратно в HBM.

***
#### Algorithm 0: Standard Attention Implementation
**Require:** Matrices $\mathbf{Q}, \mathbf{K}, \mathbf{V} \in \mathbb{R}^{N \times d}$ in HBM.


1: Load $\mathbf{Q}, \mathbf{K}$ by blocks from HBM, compute $\mathbf{S} = \mathbf{Q}\mathbf{K}^\top$, write $\mathbf{S} \in \mathbb{R}^{N \times N}$ to HBM.


2: Read $\mathbf{S}$ from HBM, compute $\mathbf{P} = \text{softmax}(\mathbf{S})$, write $\mathbf{P} \in \mathbb{R}^{N \times N}$ to HBM.


3: Load $\mathbf{P}$ and $\mathbf{V}$ by blocks from HBM, compute $\mathbf{O} = \mathbf{P}\mathbf{V}$, write $\mathbf{O} \in \mathbb{R}^{N \times d}$ to HBM.


4: **Return** $\mathbf{O}$.
***

Материализация матриц $\mathbf{S}$ и $\mathbf{P}$ в HBM приводит к выделению $O(N^2)$ памяти и многократному циклу чтения/записи массива из $N^2$ элементов, что делает весь блок сильно memory-bound.

---

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    
# Проверяем, есть ли нужная нам функция для FlashAttention
print(f"SDPA available: {hasattr(torch.nn.functional, 'scaled_dot_product_attention')}")

PyTorch version: 2.12.0+cu132
CUDA available: True
GPU Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU
SDPA available: True


In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd

# 1. Настройки размерностей (подбираем под 4GB VRAM)
B = 2          # Batch size
H = 12         # Number of attention heads
N = 4096       # Sequence length
d = 64         # Head dimension

# # 1. Настройки размерностей (подбираем под 4GB VRAM)
# B = 16         # Batch size
# H = 16         # Number of attention heads
# N = 1024       # Sequence length
# d = 64         # Head dimension

# Инициализируем тензоры в FP16 (для этого формата оптимизирован FlashAttention)
q = torch.randn(B, H, N, d, dtype=torch.float16, device='cuda')
k = torch.randn(B, H, N, d, dtype=torch.float16, device='cuda')
v = torch.randn(B, H, N, d, dtype=torch.float16, device='cuda')

def benchmark_attention(backend='standard', num_runs=50):
    # Настраиваем бэкенд
    if backend == 'standard':
        # Принудительно используем классический алгоритм (Math)
        torch.backends.cuda.enable_math_sdp(True)
        torch.backends.cuda.enable_flash_sdp(False)
        # Кстати, что это: Memory-Efficient Attention = Lazy Evaluation + Tiling
        # Используется для старых видеокарт и вычислений FP32
        # Не имеет агрессивного управления регистрами и специфического обхода кэша L1/L2
        torch.backends.cuda.enable_mem_efficient_sdp(False)
    elif backend == 'flash':
        # Принудительно используем FlashAttention
        torch.backends.cuda.enable_math_sdp(False)
        torch.backends.cuda.enable_flash_sdp(True)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        
    # Прогрев GPU (чтобы частоты поднялись до максимума)
    for _ in range(10):
        _ = F.scaled_dot_product_attention(q, k, v)
    torch.cuda.synchronize()

    # Замер времени
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    for _ in range(num_runs):
        _ = F.scaled_dot_product_attention(q, k, v)
    end_event.record()
    torch.cuda.synchronize()
    
    # Среднее время в миллисекундах
    avg_time_ms = start_event.elapsed_time(end_event) / num_runs
    return avg_time_ms

# ==========================================
# 2. АНАЛИТИЧЕСКИЙ РАСЧЕТ МЕТРИК (I/O и FLOPs)
# ==========================================
# Формулы из статьи Tri Dao. 2 байта на элемент (FP16).
bytes_per_elem = 2

# HBM R/W (ГБ)
# Standard: Читает Q,K, пишет S. Читает S, пишет P. Читает P, V, пишет O.
# Суммарно: ~ 4Nd + 4N^2 на одну голову.
hbm_standard_bytes = B * H * (4 * N * d + 4 * (N ** 2)) * bytes_per_elem
hbm_standard_gb = hbm_standard_bytes / (1024**3)

# FlashAttention: Читает Q, K, V один раз, пишет O один раз. Нет матриц N^2.
# Суммарно: ~ 4Nd на одну голову.
hbm_flash_bytes = B * H * (4 * N * d) * bytes_per_elem
hbm_flash_gb = hbm_flash_bytes / (1024**3)

# FLOPs (GFLOPs)
# Матричные умножения (QK^T и PV) требуют 4*N^2*d FLOPs.
matmul_flops = 4 * B * H * (N ** 2) * d
# В классическом Attention Softmax требует ~3 операций на элемент (exp, sum, div).
standard_flops = matmul_flops + 3 * B * H * (N ** 2)
# Во FlashAttention Softmax пересчитывается (online softmax), требуя ~5 операций на элемент.
flash_flops = matmul_flops + 5 * B * H * (N ** 2)

gflops_standard = standard_flops / (10**9)
gflops_flash = flash_flops / (10**9)

# ==========================================
# 3. ВЫПОЛНЕНИЕ БЕНЧМАРКА
# ==========================================
time_standard = benchmark_attention('standard')
time_flash = benchmark_attention('flash')

# Формируем таблицу результатов
results = pd.DataFrame({
    'Metric': ['GFLOPs', 'HBM R/W (GB)', 'Runtime (ms)'],
    'Standard': [
        f"{gflops_standard:.1f}", 
        f"{hbm_standard_gb:.3f}", 
        f"{time_standard:.1f}"
    ],
    'FA2': [
        f"{gflops_flash:.1f}", 
        f"{hbm_flash_gb:.3f}", 
        f"{time_flash:.1f}"
    ]
})

print(f"Конфигурация: B={B}, H={H}, N={N}, d={d} | GPU: RTX 3050 Ti\n")
print(results.to_string(index=False))

Конфигурация: B=2, H=12, N=4096, d=64 | GPU: RTX 3050 Ti

      Metric Standard FLASHATTENTION
      GFLOPs    104.3          105.1
HBM R/W (GB)    3.047          0.047
Runtime (ms)    433.5            6.5


### 5. Решение FlashAttention: Tiling и Online Safe Softmax

FlashAttention применяет техники **tiling** (вычисление блоками) и **kernel fusion** (слияние операций в один вызов). Матрицы $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ загружаются блоками в SRAM, где вычисляется итоговый результат $\mathbf{O}$ без промежуточного экспорта в HBM. Если бы все операции были линейными, проблем бы не возникло. Однако, наш построчный Softmax требует знания всей строки. Однако, мы все же можем вычислять его блочно, с точностью до нормиовки.

#### Математическое ограничение: нелокальность Softmax
Для предотвращения переполнения FP16 используется Safe Softmax:
$$ \text{Softmax}(x_i) = \frac{e^{x_i - m}}{\sum_{j=1}^{N} e^{x_j - m}} $$
где $m$ — максимальное значение в строке. Знаменатель требует полного прохода по строке. При блочном вычислении глобальный максимум $m$ и знаменатель изначально неизвестны.

#### Алгоритм Online Safe Softmax
Проблема решается сохранением промежуточной ненормализованной матрицы выходов и ее динамическим масштабированием при поступлении новых блоков.

Для $i$-го блока запросов $\mathbf{q}$ и $j$-го блока ключей/значений $\mathbf{K}_j, \mathbf{V}_j$:
1. Вычисляется локальная матрица $\mathbf{S}_{ij}$.
2. Находится локальный максимум: $m^{(j)} = \max(\mathbf{S}_{ij})$.
3. Обновляется глобальный максимум для текущей строки: $m^{(new)} = \max(m^{(old)}, m^{(j)})$.
4. Вычисляются поправочные коэффициенты: $e_{old} = e^{m^{(old)} - m^{(new)}}$ и $e_{new} = e^{m^{(j)} - m^{(new)}}$.

Знаменатель $\ell$ и ненормализованный выход $\mathbf{O}_{unnorm}$ корректируются с учетом нового максимума:
$$ \ell^{(new)} = \ell^{(old)} \cdot e_{old} + \sum e^{\mathbf{S}_{ij} - m^{(j)}} \cdot e_{new} $$
$$ \mathbf{O}_{unnorm}^{(new)} = \mathbf{O}_{unnorm}^{(old)} \cdot e_{old} + \left( e^{\mathbf{S}_{ij} - m^{(j)}} \mathbf{V}_j \right) \cdot e_{new} $$

По завершении прохода по строке применяется нормализация $\mathbf{O} = \mathbf{O}_{unnorm} / \ell$, после чего готовый блок пишется в HBM.

***
#### Схема алгоритма FlashAttention (Forward Pass)

<img src="main.png" width="1000">

**Require:** Blocks of $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ loaded into SRAM.

1: Initialize $\mathbf{O} = 0$, $m = -\infty$, $\ell = 0$ in SRAM/Registers.

2: **For** each block $j$ of $\mathbf{K}, \mathbf{V}$:

3: $\quad$ Load $\mathbf{K}_j, \mathbf{V}_j$ into SRAM.

4: $\quad$ Compute block scores $\mathbf{S}_{ij} = \mathbf{Q}_i \mathbf{K}_j^\top$.

5: $\quad$ Compute local max $\tilde{m}_{ij}$ and update global max $\tilde{m} = \max(m, \tilde{m}_{ij})$.

6: $\quad$ Compute scaling factors $e_{old} = e^{m - \tilde{m}}$ and $e_{new} = e^{\tilde{m}_{ij} - \tilde{m}}$.

7: $\quad$ Update denominator $\ell = \ell \cdot e_{old} + \text{sum}(e^{\mathbf{S}_{ij} - \tilde{m}_{ij}}) \cdot e_{new}$.

8: $\quad$ Update unnormalized $\mathbf{O} = \mathbf{O} \cdot e_{old} + (e^{\mathbf{S}_{ij} - \tilde{m}_{ij}} \mathbf{V}_j) \cdot e_{new}$.

9: $\quad$ $m = \tilde{m}$.

10: **End For**

11: Normalize $\mathbf{O} = \mathbf{O} / \ell$ and write to HBM.
***

#### Теоретические свойства и сложность

Алгоритм не использует аппроксимаций и возвращает математически точный результат (с точностью до ошибок машинного округления).

> **Theorem 1.** *Algorithm 1 returns $\mathbf{O} = \text{softmax}(\mathbf{Q}\mathbf{K}^\top)\mathbf{V}$ with $O(N^2 d)$ FLOPs and requires $O(N)$ additional memory beyond inputs and output.*

Ключевым преимуществом алгоритма является оптимизация I/O complexity. Сокращение числа обращений к HBM достигается за счет локализации вычислений в SRAM.

> **Theorem 2 (I/O Complexity).** *Let $M$ be the size of SRAM. Standard Attention requires $\Theta(N d + N^2)$ HBM accesses. FlashAttention requires $\Theta(N^2 d^2 / M)$ HBM accesses.*

Поскольку для стандартных архитектур $d^2 \ll M$, FlashAttention кратно снижает трафик с глобальной памятью. Возросшее количество FLOPs (из-за пересчета нормировочных коэффициентов) нивелируется устранением memory-bottleneck, что на практике приводит к сокращению времени выполнения.

**Результаты** из статьи для GPT-2 medium (seq. length 1024, head dim. 64, 16 heads, batch size 64) on A100 GPU.

| Attention | Standard | FlashAttention |
| :--- | :---: | :---: |
| GFLOPs | 66.6 | 75.2 |
| HBM R/W (GB) | 40.3 | 4.4 |
| Runtime (ms) | 41.7 | 7.3 |
